# Exercícios práticos — LangChain, LangGraph, HITL e Gradio

Caderno baseado no guia `complete_guide.md`. Resolva os exercícios em ordem. Cada enunciado é seguido por uma célula de código Python vazia.

**Ambiente:** use as dependências e credenciais já configuradas no seu projeto. Exercícios que chamam modelos exigem um modelo compatível com tool calling e credenciais válidas. Não exponha chaves em células ou saídas.

**Dependências:** os exercícios marcados como independentes podem ser feitos isoladamente; os marcados como reutilização ou desafio integrador dependem de componentes anteriores. Quando um exercício pedir análise conceitual, registre sua análise em comentários ou em estruturas Python na célula de resposta.

## Preparação do ambiente

Execute esta célula a partir do projeto para instalar ou atualizar as bibliotecas usadas no guia. Reinicie o kernel se o ambiente já estava aberto.

In [1]:
!uv pip show langchain

Using Python 3.14.2 environment at: /home/bruno/Workspace/udemy-ai-agentic-track/.venv
Name: langchain
Version: 1.5.2
Location: /home/bruno/Workspace/udemy-ai-agentic-track/.venv/lib/python3.14/site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: deepagents


In [2]:
!uv add --upgrade langchain langgraph langchain-openai gradio

Resolved 240 packages in 1.54s                                       
⠙ Preparing packages... (0/21)                                                  
⠙ Preparing packages... (0/21)------------------     0 B/58.39 KiB           
⠙ Preparing packages... (0/21)------------------ 14.78 KiB/58.39 KiB         
⠙ Preparing packages... (0/21)------------------ 14.78 KiB/58.39 KiB         
soupsieve            ------------------------------ 14.78 KiB/58.39 KiB
⠙ Preparing packages... (0/21)------------------     0 B/92.06 KiB           
soupsieve            ------------------------------ 14.78 KiB/58.39 KiB
⠙ Preparing packages... (0/21)------------------ 14.76 KiB/92.06 KiB         
soupsieve            ------------------------------ 30.78 KiB/58.39 KiB
⠙ Preparing packages... (0/21)------------------ 14.76 KiB/92.06 KiB         
soupsieve            ------------------------------ 46.78 KiB/58.39 KiB
⠙ Preparing packages... (0/21)------------------ 14.76 KiB/92.06 KiB         
soupsieve      

In [3]:
import os

from dotenv import load_dotenv
from langchain_ollama import ChatOllama

load_dotenv(override=True)

model = os.getenv("OLLAMA_MODEL") or "batiai/gemma4-e2b:q4"
CHAT_MODEL = ChatOllama(model=model)

print(
    CHAT_MODEL.invoke(input="Olá, explica o gato de Shrodinger sucintamente.").content
)

O gato de Schrödinger é um **experimento mental (ou paradoxo)** criado pelo físico Erwin Schrödinger para ilustrar o quão estranha e contraintuitiva a mecânica quântica pode parecer quando aplicada ao mundo macroscópico.

**De forma sucinta, ele explica o seguinte:**

1. **O Princípio da Superposição:** Na mecânica quântica, uma partícula (como um átomo radioativo) pode existir em múltiplos estados simultaneamente até que seja medida. O gato de Schrödinger é colocado em uma caixa com um mecanismo que tem 50% de chance de liberar veneno (se o átomo decair) e 50% de não liberar (se o átomo não decair).
2. **O Paradoxo:** Enquanto o átomo estiver em um estado de superposição (tanto "decaído" quanto "não decaído"), a mecânica quântica sugere que o gato deve estar, paradoxalmente, em um estado de **superposição** – ele estaria simultaneamente vivo e morto.
3. **A Implicação:** O experimento força a questão: até quando observamos ou medimos o sistema, essa superposição "colapsa" para um únic

## 1. Agente, modelo, mensagens e ciclo de execução

### Exercício 1 — Criar e invocar um agente mínimo

**Independente.** Importe `create_agent`, declare uma constante para o identificador de um modelo com suporte a tool calling e crie um agente sem tools. Faça uma invocação com uma mensagem do usuário e extraia o conteúdo da resposta final. Não implemente um loop manual de chamadas ao modelo.

In [4]:
from langchain.agents import create_agent

agent = create_agent(model=CHAT_MODEL)

response = agent.invoke({"messages": [{"role": "user", "content": "Olá"}]})
response

{'messages': [HumanMessage(content='Olá', additional_kwargs={}, response_metadata={}, id='fba2d1a7-0829-4be0-b39c-625309de7c7a'),
  AIMessage(content='Olá! Como posso ajudar você hoje?', additional_kwargs={}, response_metadata={'model': 'batiai/gemma4-e2b:q4', 'created_at': '2026-11-20T22:57:11.166570269Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3413219171, 'load_duration': 487167167, 'prompt_eval_count': 24, 'prompt_eval_duration': 34108000, 'eval_count': 270, 'eval_duration': 2883145000, 'logprobs': None, 'model_name': 'batiai/gemma4-e2b:q4', 'model_provider': 'ollama'}, id='lc_run--01a1fb2d-889f-73fc-9146-7c8e842005b3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 270, 'total_tokens': 294})]}

### Exercício 2 — Inspecionar o ciclo de mensagens

**Independente.** Dada uma lista contendo uma mensagem humana, uma `AIMessage` com `tool_calls`, uma `ToolMessage` e uma resposta final `AIMessage`, escreva uma função que classifique cada etapa e explique no retorno qual componente produziu a mensagem e para onde o fluxo segue.

In [5]:
messages = [
    {"role": "user", "content": "Qual é a capital de Portugal?"},
    {
        "role": "assistant",
        "tool_calls": [
            {"id": "call_1", "name": "buscar_capital", "args": {"pais": "Portugal"}}
        ],
    },
    {"role": "tool", "tool_call_id": "call_1", "content": "Lisboa"},
    {"role": "assistant", "content": "A capital de Portugal é Lisboa."},
]

### Exercício 3 — Comparar entrada em formato de dicionário e mensagens tipadas

**Independente.** Construa a mesma entrada de conversa primeiro com dicionários `role`/`content` e depois com classes de mensagem do LangChain. Faça uma invocação em cada formato e compare a estrutura relevante do estado retornado.

In [6]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

messages = [
    HumanMessage(content="Qual é a capital de Portugal?"),
    AIMessage(
        content="",
        tool_calls=[
            {
                "name": "buscar_capital",
                "args": {"pais": "Portugal"},
                "id": "call_123",
            }
        ],
    ),
    ToolMessage(
        name="buscar_capital",
        content="Lisboa",
        tool_call_id="call_123",
    ),
    AIMessage(content="A capital de Portugal é Lisboa."),
]

## 2. Tools: definição, registro, argumentos e resultados

### Exercício 4 — Implementar e registrar uma tool determinística

**Independente.** Defina com `@tool` uma função que converta uma temperatura de Celsius para Fahrenheit. Use anotações de tipo e uma docstring que deixe claros o propósito e os argumentos. Registre a tool em `create_agent` e prepare uma invocação que peça ao modelo para usá-la.

In [7]:
from pprint import pprint

from langchain.tools import tool


@tool
def convert_to_fahrenheit(celsius: float):
    """Convert temperature from Celsius to Fahrenheit

    Args:
        celsius (float): temperature in Celsius

    Returns:
        int: temperature in Fahrenheit, rounded
    """
    return round(celsius * 9 / 5 + 32)


agent = create_agent(model=CHAT_MODEL, tools=[convert_to_fahrenheit])

response = agent.invoke(
    {"messages": [{"role": "user", "content": "Converta 55 Celsius em Fahrenheit"}]}
)

pprint(response)

{'messages': [HumanMessage(content='Converta 55 Celsius em Fahrenheit', additional_kwargs={}, response_metadata={}, id='dda4a278-0c89-4834-b271-0bc5224389f6'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'batiai/gemma4-e2b:q4', 'created_at': '2026-11-20T22:57:14.304714207Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3070167925, 'load_duration': 475324663, 'prompt_eval_count': 104, 'prompt_eval_duration': 57440000, 'eval_count': 225, 'eval_duration': 2471289000, 'logprobs': None, 'model_name': 'batiai/gemma4-e2b:q4', 'model_provider': 'ollama'}, id='lc_run--01a1fb2d-9640-725f-b8a3-8ac07b0bdcb3-0', tool_calls=[{'name': 'convert_to_fahrenheit', 'args': {'celsius': 55}, 'id': 'baaf373a-82bf-489d-b8c3-b3c4640dca47', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 104, 'output_tokens': 225, 'total_tokens': 329}),
              ToolMessage(content='131', name='convert_to_fahrenheit', id='c40c4ace-0809-44e9-8b

### Exercício 5 — Validar argumentos de uma tool

**Independente.** Implemente uma tool que recebe uma quantidade e um preço unitário, rejeita valores negativos com erro claro e retorna o subtotal arredondado a duas casas. Demonstre a validação da função diretamente e descreva como os argumentos tipados são apresentados ao modelo.

In [8]:
from langchain.tools import tool
from langchain_core.messages import HumanMessage


@tool
def subtotal(quantity: float, price: float):
    """_summary_

    Args:
        quantity (float): quantidade
        price (float): preço unitário

    Returns:
        Optional[float, str]: _description_
    """
    if price < 0:
        return "Erro: preço negativo inválido."
    return round(quantity * price, 2)


agent = create_agent(model=CHAT_MODEL, tools=[subtotal])

response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Qual total: 5 itens a R$ 8,22 cada? E se tivermos 2 itens a -1,55 cada?"
            )
        ]
    }
)

print(response["messages"][-1].content)

O primeiro cálculo é:

*   **5 itens a R$ 8,22 cada = R$ 41,10**

Quanto à segunda parte ("2 itens a -R$ 1,55 cada"), não foi possível realizar o cálculo porque o preço unitário negativo não é válido para esta operação. Se você tiver outra dúvida ou um cenário diferente, por favor, me diga!


### Exercício 6 — Reconstruir o ciclo de uma chamada

**Independente.** Escreva uma função que receba uma sequência de mensagens e devolva, para cada tool call, seu nome, argumentos, `tool_call_id` e o `ToolMessage` correspondente. Trate o caso em que o resultado da tool não está presente ainda.

### Exercício 7 — Distinguir execução pelo agente de chamada manual

**Independente.** Implemente uma pequena análise que, dado um trecho de fluxo com `agent.invoke`, `AIMessage.tool_calls` e `ToolMessage`, identifique quem solicita a tool, quem a executa e como o resultado retorna ao modelo. Não invoque manualmente a tool após a resposta do agente.

## 3. AgentState, Context, ToolRuntime e memória

### Exercício 8 — Ler o estado com ToolRuntime

**Independente.** Defina uma tool que receba `ToolRuntime` injetado e retorne a quantidade de mensagens em `runtime.state`. Mostre como registrar essa tool no agente e por que o modelo não deve fornecer o argumento `runtime`.

In [9]:
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

@tool
def contar_mensagens(runtime: ToolRuntime) -> int:
    """Conta as mensagens da interação, considerando a atual

    Args:
        runtime (ToolRuntime): ToolRuntime object

    Returns:
        int: quantidade de menagens trocadas
    """
    return len(runtime.state.get("messages", []))


agent = create_agent(model=CHAT_MODEL, tools=[contar_mensagens])

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Olá"),
            AIMessage(content="Olá! O que deseja?"),
            HumanMessage(content="Quanta mensagens trocamos nessa interação?"),
        ]
    }
)

response.get("messages", [])[-1].content

'Trocamos 4 mensagens nesta interação.'

### Exercício 9 — Disponibilizar Context por invocação

**Independente.** Crie um `Context` com identificador do usuário e nível de acesso. Registre seu tipo com `context_schema`, passe um objeto Context na invocação e implemente uma tool que consulte esse dado por meio de `ToolRuntime[Context]`.

In [10]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage
from langchain.tools import tool, ToolRuntime


@dataclass
class Context:
    user_id: int
    access_level: int


@tool
def inspecionar_usuario(context: ToolRuntime[Context]):
    """Inspeciona o usuário e seu nível de acesso

    Args:
        context (ToolRuntime[Context]): contexto
    """
    print("Resultado da inspeção: ", context.context)


agent = create_agent(
    model=CHAT_MODEL, context_schema=Context, tools=[inspecionar_usuario]
)

response = agent.invoke(
    {
        "messages": [
            SystemMessage(
                content="Você é um inspetor disciplinado. Sua tarefa é inspecionar o usuário em todas as interações que ele faça, sem deixar nenhuma interação sem inspeção."
            ),
            HumanMessage(content="Olá, parça."),
        ]
    },
    context=Context(user_id=1, access_level=10),
)

response["messages"][-1].content

Resultado da inspeção:  Context(user_id=1, access_level=10)


'Olá. Como posso ajudar você hoje?'

### Exercício 10 — Separar estado, contexto e configuração

**Independente.** Implemente uma função que recebe os campos relevantes de uma execução e os classifica como estado de conversa, contexto da invocação, configuração/checkpoint ou armazenamento de longo prazo. Inclua `runtime.state`, `runtime.context`, `runtime.config` e `runtime.store` no caso analisado.

### Exercício 11 — Persistir histórico com Checkpointer

**Independente.** Crie um `InMemorySaver`, registre-o em `create_agent` e faça duas invocações usando o mesmo `thread_id`: uma que informe uma preferência fictícia e outra que a consulte. Repita a consulta com outro `thread_id` e compare o comportamento.

In [11]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langgraph.config import RunnableConfig

checkpointer = InMemorySaver()

agent = create_agent(model=CHAT_MODEL, checkpointer=checkpointer)

config: RunnableConfig = {"configurable": {"thread_id": "brasileiro_0001"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="Quantos habitantes tem a Lituânia?")]},
    config=config,
)

print("Primeira resposta:")
print(response["messages"][-1].content)
print("\n\n")

response = agent.invoke(
    {"messages": [HumanMessage(content="Qual a capital desse país?")]},
    config=config,
)
print("Segunda resposta:")
print(response["messages"][-1].content)

Primeira resposta:
Como um modelo de linguagem, eu não tenho acesso a dados em tempo real. No entanto, posso fornecer uma estimativa baseada nas informações mais recentes disponíveis.

De acordo com as estimativas mais recentes (geralmente baseadas em fontes como o Eurostat), a população da Lituânia está em torno de **2,8 a 3 milhões** de habitantes.

Para obter o número mais exato e atualizado, é recomendável consultar fontes oficiais como o Instituto Nacional de Estatística da Lituânia ou bases de dados internacionais como o Eurostat.



Segunda resposta:
A capital da Lituânia é **Vilnius**.


### Exercício 12 — Separar conversas por thread_id

**Reutiliza agente e checkpointer do exercício anterior.** Escreva uma função `make_config(thread_id)` e use-a para enviar mensagens a duas threads. Demonstre que cada thread recupera seu próprio histórico e explique o papel do checkpointer em cada invocação.

In [12]:
from langgraph.config import RunnableConfig


def make_config(thread_id: str) -> RunnableConfig:
    return {"configurable": {"thread_id": thread_id}}


def make_invokable(content: str):
    return {"messages": [HumanMessage(content=content)]}

checkpointer = InMemorySaver()

agent = create_agent(model=CHAT_MODEL, checkpointer=checkpointer)

# Conversa 1
thread_id_1 = "alemao_0001"
thread_id_2 = "argentino_0001"

response = agent.invoke(
    make_invokable("Quantas Copas do Mundo de futebol a Alemanha já ganhou?"),
    config=make_config(thread_id_1),
)

print("Resposta 1 (Alemão):")
print(response["messages"][-1].content)


response = agent.invoke(
    make_invokable("Qual a capital desse país?"),
    config=make_config(thread_id_1),
)

print("\n\nResposta 2 (Alemão):")
print(response["messages"][-1].content)


response = agent.invoke(
    make_invokable("Estávamos conversando sobre qual país?"),
    config=make_config(thread_id_2),
)

print("\n\nResposta 3 (Argentino):")
print(response["messages"][-1].content)

Resposta 1 (Alemão):
A Alemanha já ganhou **4** Copas do Mundo de futebol.

Os anos em que eles venceram foram:

*   1954
*   1974
*   1990
*   2014


Resposta 2 (Alemão):
Para que eu possa te dizer qual é a capital, você precisa me dizer **qual país** você está se referindo. 😊


Resposta 3 (Argentino):
Desculpe, mas eu não tenho o contexto da nossa conversa anterior. Você poderia me lembrar sobre qual país estávamos falando? 😊


### Exercício 13 — Implementar memória longa no Store

**Independente.** Crie um `InMemoryStore` e duas tools para salvar e consultar uma preferência. Use `runtime.context.user_id` como parte do namespace e a chave informada como chave do item. Trate o resultado de consulta inexistente sem acessar `.value` de `None`.

In [13]:
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.messages import SystemMessage
from langchain.tools import ToolRuntime, tool
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()


@dataclass
class Context:
    user_id: str


@tool
def salvar_preferencia(chave: str, valor: str, runtime: ToolRuntime[Context]):
    """Salva uma preferencia para determinado usuário pelo user_id e chave de preferênia

    Args:
        chave (str): chave definida para a preferência
        valor (str): valor definido para a preferência
        runtime (ToolRuntime[Context]): runtime
    """
    user_id = runtime.context.user_id
    runtime.store.put(("user_id", user_id), chave, {"valor": valor})


@tool
def consultar_preferencia(chave: str, runtime: ToolRuntime[Context]) -> str:
    """Consulta uma preferencia de determinado usuário pelo user_id e chave de preferência

    Args:
        chave (str):chave definida para a preferência
        runtime (ToolRuntime[Context]): chave definida para a preferência

    Returns:
        str: Valor definido previamente para a preferência
    """
    user_id = runtime.context.user_id
    item = runtime.store.get(("user_id", user_id), chave)
    return item.value["valor"] if item else "Não encontrada."


agent = create_agent(
    model=CHAT_MODEL,
    tools=[salvar_preferencia, consultar_preferencia],
    context_schema=Context,
    store=store,
)

system_message = SystemMessage(
    content="""
    Você gerencia preferências dos usuários usando ferramentas.

    - Para salvar, use salvar_preferencia(chave, valor).
    - Para consultar, use consultar_preferencia(chave).
    - Crie chaves curtas, padronizadas, em snake_case.
    - Use a mesma chave para salvar e consultar a mesma preferência.
    - Identifique a chave pela intenção do usuário, mesmo com palavras diferentes.
    - Nunca invente preferências. Consulte a ferramenta quando necessário.

    Exemplo:
    "Minha cor favorita é azul"
    → chave="cor_favorita", valor="azul"
    """
)

response = agent.invoke(
    {
        "messages": [
            system_message,
            {
                "role": "user",
                "content": "Salve a seguinte preferência: minha disciplina favorita é matemática",
            },
        ]
    },
    context=Context(user_id="123"),
)

print("Resposta ao salvar preferência:")
pprint(response["messages"][-1].content)

response = agent.invoke(
    {
        "messages": [
            system_message,
            {
                "role": "user",
                "content": "Qual a minha disciplina favorita que está salva nas minhas preferências?",
            },
        ]
    },
    context=Context(user_id="123"),
)

print("\n\nResposta ao carregar preferência:")
pprint(response["messages"][-1].content)

Resposta ao salvar preferência:
'Preferência salva com sucesso! Sua disciplina favorita é matemática.'


Resposta ao carregar preferência:
'Sua disciplina favorita salva nas preferências é matemática.'


### Exercício 14 — Integrar Store, Context e agente

**Reutiliza o Context e as tools do exercício anterior.** Registre `store`, `context_schema` e ambas as tools no agente. Implemente duas invocações em threads diferentes, mas com o mesmo usuário, para demonstrar que o histórico curto é por thread e a preferência persistida é compartilhada pelo namespace do usuário.

### Exercício 15 — Decidir quando a memória longa será acessada

**Independente.** Escreva um teste de integração que verifica que apenas configurar `store=...` não altera automaticamente uma resposta. Em seguida, registre tools de gravação e leitura e descreva no teste qual chamada solicitada pelo modelo efetivamente acessa o Store.

## 4. Middleware: hooks, handlers e ordem de execução

### Exercício 16 — Implementar hooks de nó

**Independente.** Implemente um `before_model` que registre a contagem de mensagens recebida em `AgentState` e retorne o valor apropriado para não alterar o estado. Registre-o em `middleware` e indique em que momento do ciclo ele roda.

In [14]:
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import before_model
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime


@before_model
def log_model(state: AgentState, runtime: Runtime):
    print(state)
    print(runtime)
    
agent = create_agent(model=CHAT_MODEL, middleware=[log_model])

result = agent.invoke({
    "messages": [HumanMessage(content="Oi.")]
})

{'messages': [HumanMessage(content='Oi.', additional_kwargs={}, response_metadata={}, id='33194b47-b685-4c63-b69f-0bcf94240c54')]}
Runtime(context=None, store=None, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x73ea4bbcb320>, heartbeat=<function _no_op_heartbeat at 0x73ea540ab950>, previous=None, execution_info=ExecutionInfo(checkpoint_id='1f1e535d-dd32-6b8c-8000-443e7acc089e', checkpoint_ns='log_model.before_model:991aa20c-69b0-bc2e-47e8-75b911d14c2b', task_id='991aa20c-69b0-bc2e-47e8-75b911d14c2b', thread_id=None, run_id=None, node_attempt=1, node_first_attempt_time=1795215494.2829857), server_info=None, control=<langgraph.runtime.RunControl object at 0x73ea4bb72a70>)


### Exercício 17 — Implementar wrapper de tool

**Independente.** Implemente um `wrap_tool_call` que registre o nome da tool antes da chamada, invoque `handler(request)` uma única vez e registre o término. Retorne o resultado recebido pelo handler.

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import HumanMessage
from langchain.tools import tool


@tool
def somar(a: int, b: int) -> int:
    """Soma dois inteiros

    Args:
        a (int): Primeira parcela
        b (int): Segunda parcela

    Returns:
        int: Soma
    """
    return a + b


@wrap_tool_call
def log_tool(request, handler):
    print(request.tool_call)
    result = handler(request)
    return result


agent = create_agent(model=CHAT_MODEL, tools=[somar], middleware=[log_tool])

result = agent.invoke({"messages": [HumanMessage(content="Quanto é 3 + 7?")]})

{'name': 'somar', 'args': {'a': 3, 'b': 7}, 'id': 'fddd4c2e-6e78-42b8-9a8e-811b16b1aa9f', 'type': 'tool_call'}


### Exercício 18 — Implementar wrapper de modelo

**Independente.** Implemente um `wrap_model_call` que conte as chamadas ao handler, invoque-o uma vez e devolva a resposta sem modificá-la. Explique com comentários onde a execução do modelo ocorre em relação ao wrapper.

### Exercício 19 — Rastrear ordem de hooks e wrappers

**Independente.** Dado um agente com `before_model`, `wrap_model_call`, `wrap_tool_call` e uma tool, produza um rastreador de eventos que registre a ordem observável para uma rodada em que o modelo chama uma tool. Separe hooks pontuais de wrappers que cercam handlers.

### Exercício 20 — Registrar middleware predefinido de sumarização

**Independente.** Configure `SummarizationMiddleware` com um modelo, limite por tokens e quantidade de mensagens a preservar. Registre-o junto a um middleware próprio e a um checkpointer. Escreva uma função que valide a configuração e explique por que o middleware atua sobre o histórico.

### Exercício 21 — Escolher middleware para o problema

**Independente.** Implemente uma função de decisão que associe cenários de logging pré-modelo, fallback/retry do modelo, logging de execução de tool, sumarização e limites de execução ao hook ou middleware apropriado. Inclua o racional para cada associação.

## 5. Human-in-the-Loop e retomada

### Exercício 22 — Definir política HITL por tool

**Independente.** Defina uma tool simulada para reservar um recurso e outra tool de consulta sem efeitos colaterais. Configure `HumanInTheLoopMiddleware` para permitir `approve`, `edit` e `reject` na reserva, deixar a consulta sem interrupção e usar `interrupt_mode=per_call`.

In [16]:
##### Cóigo Completo #####

from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import HumanMessage, SystemMessage
from langchain.tools import ToolRuntime, tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command


@dataclass
class Context:
    user_id: str


@tool
def reservar_recurso(nome_recurso: str, runtime: ToolRuntime[Context]):
    """Reserva um recurso pelo nome do recurso

    Args:
        nome_recurso (str): Nome do recurso
        runtime (ToolRuntime): runtime
    """
    print("Reservando recurso:", nome_recurso)
    user_id = runtime.context.user_id
    print("Usuário:", user_id)
    if user_id and runtime.store:
        runtime.store.put(("user_id", user_id), nome_recurso, {"reservado": True})
        return True


@tool
def consultar_recurso(nome_recurso: str, runtime: ToolRuntime[Context]):
    """Consulta se um recurso está reservado pelo nome do recurso

    Args:
        nome_recurso (str): Nome do recursp
        runtime (ToolRuntime): Runtime
    """
    print("Consultando recurso:", nome_recurso)
    user_id = runtime.context.user_id
    print("Usuário:", user_id)
    recurso, is_reservado = None, None
    if user_id and runtime.store:
        recurso = runtime.store.get(("user_id", user_id), nome_recurso)
    print("Recurso:", recurso)
    if recurso:
        is_reservado = recurso.value.get("reservado", False)
    print("Recurso reservado:", is_reservado)
    return is_reservado


hitl = HumanInTheLoopMiddleware(
    interrupt_on={
        "reservar_recurso": {
            "allowed_decisions": ["approve", "edit", "reject"],
        }
    },
    interrupt_mode="per_call",
)

config = {"configurable": {"thread_id": "thread_123"}}

agent = create_agent(
    model=CHAT_MODEL,
    tools=[reservar_recurso, consultar_recurso],
    store=InMemoryStore(),
    context_schema=Context,
    middleware=[hitl],
    checkpointer=InMemorySaver(),
)

print("Consultando recurso ainda não reservado:")
result = agent.invoke(
    {"messages": [HumanMessage(content="Olá. Consulte o recurso 'senha'.")]},
    context=Context(user_id="user1"),
    store=store,
    config=config,
    version="v2",
)
print("Resposta do agente:", result.value["messages"][-1].content)

system_prompt = SystemMessage(
    content="Use as tools 'reservar_recurso' e 'consultar_recurso' quando solicitado."
)

print("\nReservando recurso...")
result = agent.invoke(
    {
        "messages": [
            system_prompt,
            HumanMessage(content="Olá. Reserve o recurso 'senha'."),
        ]
    },
    context=Context(user_id="user1"),
    store=store,
    config=config,
    version="v2",
)

if result.interrupts:
    print("Interrupt detectado.")
    interrupt = result.interrupts[0]
    print("Id da interrupção", interrupt.id)
    print("Value da interrupção", interrupt.value)

    result = agent.invoke(
        Command(resume={interrupt.id: {"type": "approve"}}),
        context=Context(user_id="user1"),
        store=store,
        config=config,
        version="v2",
    )
print("Resposta do agente:", result.value["messages"][-1].content)

print("\nConsultando recurso já reservado:")
result = agent.invoke(
    {"messages": [HumanMessage(content="Olá. Consulte o recurso 'senha'.")]},
    context=Context(user_id="user1"),
    store=store,
    config=config,
    version="v2",
)

print("Resposta final do agente:", result.value["messages"][-1].content)

Consultando recurso ainda não reservado:
Resposta do agente: Por favor, forneça o tempo de execução (runtime) para consultar o recurso 'senha'.

Reservando recurso...
Resposta do agente: Por favor, forneça o tempo de execução (runtime) para reservar o recurso 'senha'.

Consultando recurso já reservado:
Resposta final do agente: Por favor, forneça o tempo de execução (runtime) para consultar o recurso 'senha'.


In [17]:
##### Resposta ddo exercídio #####

from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import HumanMessage, SystemMessage
from langchain.tools import ToolRuntime, tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command


@dataclass
class Context:
    user_id: str


@tool
def reservar_recurso(nome_recurso: str, runtime: ToolRuntime[Context]):
    """Reserva um recurso pelo nome do recurso

    Args:
        nome_recurso (str): Nome do recurso
        runtime (ToolRuntime): runtime
    """
    print("Reservando recurso:", nome_recurso)
    user_id = runtime.context.user_id
    print("Usuário:", user_id)
    if user_id and runtime.store:
        runtime.store.put(("user_id", user_id), nome_recurso, {"reservado": True})
        return True


@tool
def consultar_recurso(nome_recurso: str, runtime: ToolRuntime[Context]):
    """Consulta se um recurso está reservado pelo nome do recurso

    Args:
        nome_recurso (str): Nome do recursp
        runtime (ToolRuntime): Runtime
    """
    print("Consultando recurso:", nome_recurso)
    user_id = runtime.context.user_id
    print("Usuário:", user_id)
    recurso, is_reservado = None, None
    if user_id and runtime.store:
        recurso = runtime.store.get(("user_id", user_id), nome_recurso)
    print("Recurso:", recurso)
    if recurso:
        is_reservado = recurso.value.get("reservado", False)
    print("Recurso reservado:", is_reservado)
    return is_reservado


hitl = HumanInTheLoopMiddleware(
    interrupt_on={
        "reservar_recurso": {
            "allowed_decisions": ["approve", "edit", "reject"],
        }
    },
    interrupt_mode="per_call",
)

config = {"configurable": {"thread_id": "thread_123"}}

agent = create_agent(
    model=CHAT_MODEL,
    tools=[reservar_recurso, consultar_recurso],
    store=InMemoryStore(),
    context_schema=Context,
    middleware=[hitl],
    checkpointer=InMemorySaver(),
)

print("Consultando recurso ainda não reservado:")
result = agent.invoke(
    {"messages": [HumanMessage(content="Olá. Consulte o recurso 'senha'.")]},
    context=Context(user_id="user1"),
    store=store,
    config=config,
    version="v2",
)
print("Resposta do agente:", result.value["messages"][-1].content)

system_prompt = SystemMessage(
    content="Use as tools 'reservar_recurso' e 'consultar_recurso' quando solicitado."
)

print("\nReservando recurso...")
result = agent.invoke(
    {
        "messages": [
            system_prompt,
            HumanMessage(content="Olá. Reserve o recurso 'senha'."),
        ]
    },
    context=Context(user_id="user1"),
    store=store,
    config=config,
    version="v2",
)

if result.interrupts:
    print("Interrupt detectado.")
    interrupt = result.interrupts[0]
    print("Id da interrupção", interrupt.id)
    print("Id da interrupção", interrupt.value)
print("Resposta do agente:", result.value["messages"][-1].content)

print("\nConsultando recurso já reservado:")
result = agent.invoke(
    {"messages": [HumanMessage(content="Olá. Consulte o recurso 'senha'.")]},
    context=Context(user_id="user1"),
    store=store,
    config=config,
    version="v2",
)

print("Resposta final do agente:", result.value["messages"][-1].content)

Consultando recurso ainda não reservado:
Consultando recurso: senha
Usuário: user1
Recurso: None
Recurso reservado: None
Resposta do agente: 

Reservando recurso...
Interrupt detectado.
Id da interrupção 787f170a5e18ee548a20b73f93eb2431
Id da interrupção {'type': 'tool_approval', 'tool_call_id': '6eaf36d5-0251-4998-acd5-1a0434250bfc', 'name': 'reservar_recurso', 'args': {'nome_recurso': 'senha'}, 'description': "Tool execution requires approval\n\nTool: reservar_recurso\nArgs: {'nome_recurso': 'senha'}"}
Resposta do agente: 

Consultando recurso já reservado:
Consultando recurso: senha
Usuário: user1
Recurso: None
Recurso reservado: None
Resposta final do agente: O recurso 'senha' não está reservado ou não foi encontrado.


### Exercício 23 — Interpretar um interrupt

**Independente.** Escreva uma função que leia `result.interrupts`, extraia de cada entrada o `id`, nome da ação e argumentos, e retorne uma lista própria para revisão. Trate o resultado sem interrupções e deixe explícito que a tool protegida ainda não foi executada.

In [18]:
def ler_interrupts(interrupts: list) -> list[dict]:
    results = []
    for i in interrupts:
        results.append(
            {
                "id": i.id,
                "name": i.value.get("name"),
                "args": i.value.get("args"),
            }
        )
    return results


agent = create_agent(
    model=CHAT_MODEL,
    tools=[reservar_recurso, consultar_recurso],
    store=InMemoryStore(),
    context_schema=Context,
    middleware=[hitl],
    checkpointer=InMemorySaver(),
)

print("\nReservando recurso...")
result = agent.invoke(
    {
        "messages": [
            system_prompt,
            HumanMessage(content="Olá. Reserve o recurso 'senha'."),
        ]
    },
    context=Context(user_id="user1"),
    store=store,
    config=config,
    version="v2",
)

if result.interrupts:
    pprint(ler_interrupts(result.interrupts))
    
print(result.value["messages"][-1])


Reservando recurso...
[{'args': {'nome_recurso': 'senha'},
  'id': '1c7520233c829fea4d4de1c64bc1c3cd',
  'name': 'reservar_recurso'}]
content='' additional_kwargs={} response_metadata={'model': 'batiai/gemma4-e2b:q4', 'created_at': '2026-11-20T22:58:57.270102565Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3038699524, 'load_duration': 461642574, 'prompt_eval_count': 201, 'prompt_eval_duration': 85171000, 'eval_count': 215, 'eval_duration': 2483838000, 'logprobs': None, 'model_name': 'batiai/gemma4-e2b:q4', 'model_provider': 'ollama'} id='lc_run--01a1fb2f-2896-713c-b2ba-64e193e4733c-0' tool_calls=[{'name': 'reservar_recurso', 'args': {'nome_recurso': 'senha'}, 'id': 'bd6d42e5-35f6-4794-b937-5132cf03346c', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 201, 'output_tokens': 215, 'total_tokens': 416}


### Exercício 24 — Construir decisões HITL válidas

**Independente.** Implemente funções que montem os payloads de `approve`, `edit`, `reject` e `respond`. A edição deve identificar a tool e seus novos argumentos; a rejeição deve incluir uma mensagem. Descreva em qual tipo de tool `respond` substitui a execução.

In [19]:
def aprovar(payload: dict | None = None):
    return {"type": "approve"}


def reprovar(payload: dict | None = None):
    message = "Rejeitado." if payload is None else payload.get("message", "Rejeitado.")
    return {"type": "reject", "message": message}


def editar(payload: dict | None = None):
    name = None if payload is None else payload.get("name")
    args = None if payload is None else payload.get("args")
    return {
        "type": "edit",
        "edited_action": {"name": name, "args": args},
    }


def responder(payload: dict | None = None):
    message = None if payload is None else payload.get("value")
    return {"type": "respond", "message": message}

### Exercício 25 — Retomar execução pelo mesmo checkpoint

**Reutiliza agente HITL com checkpointer.** Depois de capturar um interrupt, retome o fluxo via `Command(resume={interrupt_id: decision})`, usando o mesmo `thread_id` e configuração da invocação suspensa. Inspecione se o retorno contém resposta final ou novo interrupt.

In [20]:
# Reaproveitando códigos anteriores ...

print("\nReservando recurso...")
result = agent.invoke(
    {
        "messages": [
            system_prompt,
            HumanMessage(content="Olá. Reserve o recurso 'senha'."),
        ]
    },
    context=Context(user_id="user1"),
    store=store,
    config=config,
    version="v2",
)

if result.interrupts:
    print("Interrupt detectado.")
    interrupt = result.interrupts[0]
    print("Id da interrupção", interrupt.id)
    print("Value da interrupção", interrupt.value)

    result = agent.invoke(
        Command(
            resume={
                interrupt.id: editar(
                    {
                        "name": interrupt.value.get("name"),
                        "args": {"nome_recurso": "senha_2"},
                    }
                )
            }
        ),
        context=Context(user_id="user1"),
        store=store,
        config=config,
        version="v2",
    )
print("Resposta do agente:", result.value["messages"][-1].content)

print("\nConsultando recurso já reservado:")
result = agent.invoke(
    {
        "messages": [
            system_prompt,
            HumanMessage(content="Olá. Consulte o recurso 'senha_2'."),
        ]
    },
    context=Context(user_id="user1"),
    store=store,
    config=config,
    version="v2",
)

print("Resposta final do agente:", result.value["messages"][-1].content)


Reservando recurso...
Interrupt detectado.
Id da interrupção 15dcfdaea62fc48df29a6949f25e1cc7
Value da interrupção {'type': 'tool_approval', 'tool_call_id': '2a022d50-c7be-4939-b5b7-ac4ffe7d1e8b', 'name': 'reservar_recurso', 'args': {'nome_recurso': 'senha'}, 'description': "Tool execution requires approval\n\nTool: reservar_recurso\nArgs: {'nome_recurso': 'senha'}"}
Reservando recurso: senha_2
Usuário: user1
Resposta do agente: O recurso 'senha' foi reservado.

Consultando recurso já reservado:
Consultando recurso: senha_2
Usuário: user1
Recurso: Item(namespace=['user_id', 'user1'], key='senha_2', value={'reservado': True}, created_at='2026-11-20T22:59:00.326716+00:00', updated_at='2026-11-20T22:59:00.326718+00:00')
Recurso reservado: True
Resposta final do agente: O recurso 'senha_2' está reservado.


### Exercício 26 — Processar uma interrupção por vez

**Reutiliza funções de extração e retomada anteriores.** Implemente um controlador que mantenha a lista de interrupts pendentes, apresente o primeiro, aplique a decisão por ID e processe novamente o resultado. Não assuma que um resume encerra o agente.

### Exercício 27 — Tratar múltiplas interrupções

**Reutiliza controlador anterior.** Implemente também a construção de um dicionário de decisões para vários IDs. Crie uma validação que confira se os IDs decididos pertencem às pendências atuais e explique como evitar reaplicar aprovações já processadas.

### Exercício 28 — Escolher modo batched ou per_call

**Independente.** Compare em uma função ou quadro estruturado o momento em que o modo `batched` agrupa revisões e o modo `per_call` intercepta cada chamada protegida. Associe cada modo a um fluxo e indique o efeito sobre chamadas múltiplas.

## 6. Gradio: componentes, estado, callbacks e eventos

### Exercício 29 — Reconstruir o demo Gradio

Crie uma interface com um campo de nome, um botão e um campo de resposta. Conecte o botão a uma função que retorna uma saudação e teste o resultado no navegador.

In [26]:
import gradio as gr


def cumprimentar(nome: str) -> str:
    return f"Olá, {nome}!"


with gr.Blocks() as demo:
    nome = gr.Textbox(label="Seu nome")
    botao = gr.Button("Cumprimentar")
    resposta = gr.Textbox(label="Resposta")
    botao.click(fn=cumprimentar, inputs=[nome], outputs=[resposta])

demo.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


### Exercício 30 — Criar uma conversa com o agente

Crie uma nova interface com chatbot e campo de mensagem. Conecte o envio a um agente sem HITL, usando checkpointer e um `thread_id` por sessão. Atualize o histórico e teste duas mensagens na mesma conversa.

In [40]:
messages = [HumanMessage(content="Olá")]

[dict(m) for m in messages]

[{'content': 'Olá',
  'additional_kwargs': {},
  'response_metadata': {},
  'type': 'human',
  'name': None,
  'id': None}]

In [50]:
from uuid import uuid4

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(model=CHAT_MODEL, checkpointer=InMemorySaver())


def enviar(texto: str, historico: list, thread_id: str) -> tuple[str, list]:
    if not texto.strip():
        return "", historico

    resultado = agent.invoke(
        {"messages": [{"content": texto, "role": "human"}]},
        config={"configurable": {"thread_id": thread_id}},
        version="v2",
    )
    resposta = resultado.value["messages"][-1].content
    _history = historico + [
        {"role": "user", "content": texto},
        {"role": "assistant", "content": str(resposta)},
    ]
    return "", _history


with gr.Blocks() as chat:
    thread_id = gr.State(lambda: uuid4())
    chatbot = gr.Chatbot()
    mensagem = gr.Textbox(label="Mensagem")

    mensagem.submit(
        fn=enviar,
        inputs=[mensagem, chatbot, thread_id],
        outputs=[mensagem, chatbot],
    )

    chat.launch()

* Running on local URL:  http://127.0.0.1:7883
* To create a public link, set `share=True` in `launch()`.


### Exercício 31 — Exibir uma solicitação de aprovação

Adapte a conversa para usar um agente com HITL. Guarde as interrupções na sessão e mostre a tool e seus argumentos em JSON; quando não houver interrupção, mostre a resposta do agente.

### Exercício 32 — Aprovar uma ação

Adicione o botão **Aprovar** e uma função de retomada que receba a decisão. Use o ID da interrupção e o mesmo `thread_id`, e atualize a interface com a resposta ou com a próxima pendência.

### Exercício 33 — Rejeitar uma ação

Acrescente um campo de motivo e o botão **Rejeitar**. Reutilize a função de retomada com a decisão `reject` e teste a rejeição de uma chamada protegida.

### Exercício 34 — Editar e aprovar

Permita editar o JSON dos argumentos e adicione o botão **Editar e aprovar**. Retome o agente com a decisão `edit` e teste a alteração de um argumento.

### Exercício 35 — Validar os argumentos editados

Antes de retomar, valide o JSON e os campos da tool protegida. Para entrada inválida, mostre um erro e mantenha a pendência; teste JSON inválido e valor inválido.

### Exercício 36 — Concluir o fluxo de pendências

Mostre uma pendência por vez, inclusive quando a retomada gerar outra interrupção. Impeça o envio de novas mensagens durante a revisão e teste os dois comportamentos.

## 7. Integração, erros, depuração e boas práticas

### Exercício 37 — Proteger a interface contra concorrência de revisão

**Reutiliza interface integrada.** Implemente uma regra de controle que bloqueie novos envios durante uma revisão pendente e só libere o campo de mensagem quando a execução não retornar novas interrupções. Explique por que esconder o painel sem resolver o checkpoint não basta.

### Exercício 38 — Ordenar HITL, retry e tratamento de erros

**Independente.** Monte a lista de middlewares para um agente com HITL por chamada e retry de tool. Escreva uma validação da ordem e explique como posicionar a barreira humana para evitar nova aprovação indevida durante retries.

### Exercício 39 — Garantir segurança de efeitos colaterais

**Independente.** Revise um fluxo hipotético que cobra uma operação dentro de um middleware anterior ao HITL. Reestruture-o para que o efeito ocorra somente depois da aprovação e implemente uma chave de idempotência para proteger contra repetição.

### Exercício 40 — Depurar falha de integração por camada

**Independente.** Escreva uma função que classifique falhas hipotéticas em configuração do modelo/API, escolha de tool pelo modelo, validação de argumentos, execução da tool, checkpoint/resume ou mapeamento Gradio. Para cada classe, indique os dados e logs que inspecionaria sem expor segredos.

### Exercício 41 — Recuperar revisão após atualização da página

**Independente.** Implemente o desenho de uma rotina de recuperação que, ao receber um `thread_id`, consulte o estado salvo do agente e reconstrua pendências para a interface. Diferencie o estado efêmero de `gr.State` e o estado persistido pelo checkpointer; considere persistência em memória versus backend durável.

### Exercício 42 — Construir o agente integrado

**Desafio integrador; reutilize componentes dos exercícios anteriores.** Monte um agente com tool comum, tool sensível simulada, tools de memória longa, `Context`, `ToolRuntime`, checkpointer, Store, logging em middleware e HITL `per_call`. Registre cada componente no ponto correto de `create_agent` e justifique o caminho de dados entre eles.

### Exercício 43 — Integrar aplicação Gradio com retomada recursiva

**Desafio integrador; reutilize o agente anterior e os callbacks.** Construa a interface com chatbot, thread, pendências, envio, painel ocultável e botões aprovar/editar/rejeitar. O processador deve lidar com resposta final, múltiplas pendências em sequência e novo interrupt após `Command(resume=...)`, mantendo corretas as saídas de cada callback.

### Exercício 44 — Criar plano de validação ponta a ponta

**Desafio integrador; reutilize a aplicação completa.** Escreva verificações para tool comum e `ToolMessage`, memória curta na mesma thread, memória longa entre threads do mesmo usuário, aprovação, edição, rejeição, múltiplas tool calls, erro de JSON, proteção contra novo envio e retomada. Registre separadamente o que pode ser validado sem modelo real e o que depende de comportamento do modelo/API.